In [19]:
# Task 2: Prediction of Store Sales
# 2.1 Preprocessing	
# 1.	Converting Non-Numeric Columns
# o	Use pd.get_dummies() for categorical variables.
# o	Map datetime columns to numeric features (e.g., convert days into weekdays).
# 2.	Handling NaN Values
# o	Replace or interpolate missing values:
# 	Numerical columns: Use median/mean imputation or interpolate.
# 	Categorical columns: Fill with mode or a placeholder like 'Unknown'.

In [20]:
# 3.	Feature Engineering
# o	Extract features from the datetime column:

import pandas as pd

# create a dataframe with sample data
train = pd.read_csv(r"C:\Users\Almazt\OneDrive - Ethiopian Airlines\Desktop\10 Academy\Rossmann Pharmaceuticals\Week-4\data\raw\train_cleaned_data.csv")

df = pd.DataFrame({'Date': pd.date_range(start='1/1/2021', periods=10, freq='D')})

df['Weekday'] = df['Date'].dt.weekday
df['Weekend'] = (df['Weekday'] >= 5).astype(int)
# Define a list of holidays
holidays_list = pd.to_datetime(['2021-01-01', '2021-01-06', '2021-01-20'])

# Define the function to calculate days to the next holiday
def calculate_days_to_holiday(dates, holidays):
	return dates.apply(lambda date: (holidays - date).min().days)

# Define the function to calculate days after the last holiday
def calculate_days_after_holiday(dates, holidays):
	return dates.apply(lambda date: (date - holidays[holidays <= date]).min().days)

df['Days_To_Holiday'] = calculate_days_to_holiday(df['Date'], holidays_list)
df['Days_After_Holiday'] = calculate_days_after_holiday(df['Date'], holidays_list)
# Define the function to classify the month section
def month_section_classifier(date):
	day = date.day
	if day <= 10:
		return 'Beginning'
	elif day <= 20:
		return 'Middle'
	else:
		return 'End'

df['Month_Section'] = df['Date'].apply(month_section_classifier)
# o	Suggested extra features:
# 	Public holidays (binary).
# 	Previous day’s sales or rolling averages.
# 	Lag variables (e.g., sales from 1, 7, and 30 days prior).


C:\Users\Almazt\AppData\Local\Temp\ipykernel_21696\1785142651.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r"C:\Users\Almazt\OneDrive - Ethiopian Airlines\Desktop\10 Academy\Rossmann Pharmaceuticals\Week-4\data\raw\train_cleaned_data.csv")


In [21]:
# 4.	Scaling
# o	Use StandardScaler:

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Exclude the 'Date' column from scaling
df_to_scale = df.drop(columns=['Date'])

# Convert categorical column 'Month_Section' to dummy variables
df_to_scale = pd.get_dummies(df_to_scale, columns=['Month_Section'])

scaled_features = scaler.fit_transform(df_to_scale)

In [22]:
# 2.2 Building Models with sklearn Pipelines
# •	Example using RandomForestRegressor:

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

num_cols = ['Days_To_Holiday']
cat_cols = ['Store_Type', 'Assortment']

# Add sample categorical columns for demonstration purposes
df['Store_Type'] = ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B']
df['Assortment'] = ['Basic', 'Extended', 'Basic', 'Extended', 'Basic', 'Extended', 'Basic', 'Extended', 'Basic', 'Extended']

transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(), cat_cols)
    ])

pipeline = Pipeline([
    ('preprocess', transformer),
    ('model', RandomForestRegressor())
])

# Create a sample 'Sales_Lag_7' column for demonstration purposes
df['Sales_Lag_7'] = [100, 150, 200, 250, 300, 350, 400, 450, 500, 550]

# Define the target variable 'y' and feature matrix 'X'
y = df['Sales_Lag_7']  # Assuming 'Sales_Lag_7' is the target variable
X = df.drop(columns=['Sales_Lag_7'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Days_To_Holiday']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Store_Type',
                                                   'Assortment'])])),
                ('model', RandomForestRegressor())])

In [23]:
# 2.3 Choosing a Loss Function
# •	Use Mean Absolute Error (MAE):
# o	Reason: MAE is less sensitive to outliers compared to Mean Squared Error (MSE), and it's intuitive.

from sklearn.metrics import mean_absolute_error

# Define predictions
predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
# •	Defend Choice: MAE reflects the average absolute difference between predicted and actual sales, making it an interpretable metric for stakeholders.


In [24]:
# 2.4 Post Prediction Analysis
# 1.	Feature Importance
# o	Use RandomForest.feature_importances_ to rank features by their impact.

importance = pipeline.named_steps['model'].feature_importances_
feature_names = transformer.get_feature_names_out()
feature_importance = pd.Series(importance, index=feature_names)

In [25]:
import numpy as np

# 2.	Confidence Interval Estimation
# o	Perform bootstrapping or calculate prediction intervals using percentiles:

predictions = [pipeline.predict(X_test) for _ in range(100)]
lower_bound = np.percentile(predictions, 5, axis=0)
upper_bound = np.percentile(predictions, 95, axis=0)

In [26]:
# 2.5 Serialize Models
# •	Use joblib:

import joblib
from datetime import datetime

timestamp = datetime.now().strftime('%d-%m-%Y-%H-%M-%S-%f')
filename = f"model-{timestamp}.pkl"
joblib.dump(pipeline, filename)

['model-13-01-2025-23-00-39-638143.pkl']

In [32]:
# 2.6 Building a Deep Learning Model
# 1.	Preprocess data (similar to sklearn).
# 2.	Create sliding windows for supervised learning.
# 3.	Build LSTM Model with TensorFlow:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Drop the 'Date' column from X_train and X_test
X_train_lstm = X_train.drop(columns=['Date'])
X_test_lstm = X_test.drop(columns=['Date'])

# Convert categorical columns to dummy variables
X_train_lstm = pd.get_dummies(X_train_lstm, columns=['Month_Section', 'Store_Type', 'Assortment'])
X_test_lstm = pd.get_dummies(X_test_lstm, columns=['Month_Section', 'Store_Type', 'Assortment'])

# Ensure both train and test sets have the same columns
X_test_lstm = X_test_lstm.reindex(columns=X_train_lstm.columns, fill_value=0)

# Convert boolean columns to integers
X_train_lstm = X_train_lstm.astype(int)
X_test_lstm = X_test_lstm.astype(int)

# Define time_steps and n_features based on your data
time_steps = 1  # Example value, adjust as needed
n_features = X_train_lstm.shape[1]

# Reshape the data to 3D array as required by LSTM
X_train_lstm = X_train_lstm.values.reshape((X_train_lstm.shape[0], time_steps, n_features))
X_test_lstm = X_test_lstm.values.reshape((X_test_lstm.shape[0], time_steps, n_features))

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(time_steps, n_features)),
    LSTM(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mae')
model.fit(X_train_lstm, y_train, validation_data=(X_test_lstm, y_test), epochs=20, batch_size=32)


Epoch 1/20


c:\Users\Almazt\AppData\Local\anaconda3\envs\myenv\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 325.0039 - val_loss: 324.9907
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 324.9861 - val_loss: 324.9717
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - loss: 324.9682 - val_loss: 324.9526
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - loss: 324.9501 - val_loss: 324.9332
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 324.9319 - val_loss: 324.9136
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 324.9134 - val_loss: 324.8936
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - loss: 324.8945 - val_loss: 324.8731
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 324.8752 - val_loss: 324.8521
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 324.8555 - val_loss: 324.8305
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - loss: 324.8351 - val_loss: 324.8081
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 324.8141 - val_loss: 324.7849
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/ste